In [24]:
import networkx as nx
import random
from gensim.models import Word2Vec
from node2vec import Node2Vec

/Users/visuworks/Documents/00_project/SmallTalk2Rec_Exp/my_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# 그래프 생성
def generate_random_graph(num_nodes, num_edges):
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    
    while G.number_of_edges() < num_edges:
        print(G.number_of_edges())
        u = random.randint(0, num_nodes - 1)
        v = random.randint(0, num_nodes - 1)
        if u != v:  # Self-loop 방지
            G.add_edge(u, v)

    return G


In [16]:
# 파라미터 설정
num_nodes = 10  # 노드 수
num_edges = 15  # 간선 수

# 그래프 생성
G = generate_random_graph(num_nodes, num_edges)
print("Generated Graph:")
print(G.edges())

0
0
1
1
2
3
4
5
5
6
7
8
9
9
10
11
12
12
13
14
Generated Graph:
[(0, 4), (0, 8), (0, 2), (1, 2), (1, 6), (1, 9), (1, 8), (2, 9), (2, 5), (2, 6), (2, 8), (4, 8), (4, 9), (5, 7), (7, 9)]


In [20]:



# DeepWalk 알고리즘 적용
def deepwalk(graph, num_walks=10, walk_length=5):
    walks = []
    for node in graph.nodes():
        for _ in range(num_walks):
            walk = [str(node)]  # 노드 ID를 문자열로 변환
            while len(walk) < walk_length:
                cur = walk[-1]
                neighbors = list(graph.neighbors(int(cur)))  # 문자열에서 정수로 변환
                if neighbors:
                    walk.append(str(random.choice(neighbors)))  # 선택된 이웃도 문자열로 변환
                else:
                    break
            walks.append(walk)
    return walks

In [21]:

# DeepWalk 적용
walks = deepwalk(G, num_walks=5, walk_length=5)
print("\nGenerated Walks:")
print(walks)


Generated Walks:
[['0', '2', '6', '1', '9'], ['0', '8', '2', '6', '2'], ['0', '4', '9', '7', '9'], ['0', '4', '0', '4', '8'], ['0', '2', '5', '7', '9'], ['1', '2', '0', '2', '0'], ['1', '6', '2', '5', '2'], ['1', '8', '0', '8', '2'], ['1', '8', '1', '2', '1'], ['1', '9', '2', '1', '2'], ['2', '0', '2', '6', '1'], ['2', '9', '4', '8', '1'], ['2', '1', '6', '1', '9'], ['2', '1', '6', '1', '9'], ['2', '8', '1', '6', '2'], ['3'], ['3'], ['3'], ['3'], ['3'], ['4', '8', '1', '9', '7'], ['4', '8', '0', '2', '8'], ['4', '9', '1', '6', '1'], ['4', '9', '2', '6', '1'], ['4', '9', '2', '1', '8'], ['5', '2', '0', '8', '1'], ['5', '2', '8', '4', '0'], ['5', '7', '9', '1', '9'], ['5', '2', '6', '1', '9'], ['5', '7', '9', '4', '0'], ['6', '2', '6', '2', '9'], ['6', '2', '1', '6', '2'], ['6', '2', '0', '4', '9'], ['6', '2', '6', '1', '8'], ['6', '2', '1', '8', '1'], ['7', '9', '1', '8', '0'], ['7', '5', '2', '8', '0'], ['7', '5', '2', '8', '0'], ['7', '5', '7', '5', '7'], ['7', '5', '2', '5', '7'], [

In [22]:


# Word2Vec 모델 훈련
def train_word2vec(walks):
    model = Word2Vec(sentences=walks, vector_size=64, window=5, min_count=1, sg=1)
    return model


# Word2Vec 모델 훈련
model = train_word2vec(walks)

# 노드의 임베딩 확인
for node in G.nodes():
    print(f"Node {node} embedding: {model.wv[str(node)]}")


Node 0 embedding: [-2.90841772e-03 -6.82633277e-03 -1.00966804e-02 -5.74898440e-03
  6.65347697e-03 -5.92573686e-03  1.31491972e-02  2.50482117e-03
 -1.14036324e-02  1.47531247e-02  1.20166251e-02  8.61102622e-03
 -1.07331872e-02  9.18407645e-03  6.23865519e-03  8.09373055e-03
  6.66318601e-03  3.09031783e-03 -5.02910139e-03  1.30307321e-02
  1.51365437e-02  5.97791839e-03 -4.34534252e-03 -7.76050001e-06
  1.98599277e-03 -1.32030053e-02 -1.29179731e-02 -3.52282688e-04
  1.88580598e-03 -9.06497985e-03 -7.45953480e-03 -1.15492512e-02
  1.31114917e-02  8.82106906e-05 -7.05342833e-03  8.97698849e-03
  1.43944519e-02 -6.43219845e-03  1.25400610e-02  8.35711136e-03
  9.15965997e-03  8.35947809e-04  1.27719967e-02 -1.10190529e-02
 -1.29052578e-02  1.45522160e-02 -3.67828761e-04 -3.05479695e-03
  7.22381892e-03 -6.35184720e-03  4.28338768e-03  1.09299645e-02
  9.51860379e-03 -1.17093306e-02  1.47586744e-02  7.27941701e-03
  6.21203845e-03 -9.86154936e-03  1.32609336e-02 -3.33024981e-03
  1.382

In [25]:
# Node2Vec 모델 생성
node2vec = Node2Vec(graph=G, dimensions=64, walk_length=10, num_walks=100, p=0.5, q=2, workers=4)

# 모델 훈련
model = node2vec.fit(window=5, min_count=1, batch_words=4)

# 노드의 임베딩 확인
for node in G.nodes():
    print(f"Node {node} embedding: {model.wv[str(node)]}")  # 노드 ID를 문자열로 변환

Computing transition probabilities: 100%|██████████| 10/10 [00:00<00:00, 12768.05it/s]


Node 0 embedding: [-0.04956444 -0.2034216   0.24301253  0.20615692 -0.14816037 -0.27574903
  0.10478566  0.18898372 -0.19991434 -0.07687032  0.34348878  0.06280756
  0.04070256 -0.07203659 -0.07067785 -0.07009003 -0.00786516  0.1525474
 -0.07453055  0.08541423  0.16258343  0.22136143  0.25333765 -0.04093262
  0.15349479  0.23698997 -0.07656185  0.12189797 -0.05107662 -0.06603093
 -0.04940132 -0.15127535 -0.14395954 -0.2947946  -0.0212709   0.08126824
  0.01006865 -0.02620796  0.4287434  -0.08310343 -0.00248034  0.12826811
 -0.07513759 -0.10090183  0.06014439 -0.07814574 -0.06463601 -0.08343925
  0.03875374 -0.03840011  0.23164667  0.03581364 -0.0534595   0.26909468
  0.13104284 -0.18074326 -0.11734378 -0.24943575  0.02566267  0.06989852
  0.0911978  -0.0999312  -0.17296244  0.04604968]
Node 1 embedding: [-0.06040754 -0.1825209   0.2177939   0.18188275 -0.1325981  -0.27282673
  0.08606137  0.1764433  -0.20631653 -0.11986051  0.3451826   0.06942727
  0.0764236  -0.08810313 -0.08556783 -0

Generating walks (CPU: 1): 100%|██████████| 25/25 [00:00<00:00, 8484.31it/s]

Generating walks (CPU: 3): 100%|██████████| 25/25 [00:00<00:00, 7739.14it/s]
